# Hospital Mortality Prediction (HOSPITAL_EXPIRE_FLAG)
## Computational Machine Learning - Final Project 2025

**Author:** Corneel Van Den Bosch  
**Date:** December 2025  
**Task:** Binary Classification - Predicting ICU Patient Mortality

---

## Table of Contents

1. [Introduction & Problem Statement](#1-introduction)
2. [Setup & Imports](#2-setup)
3. [Data Loading & Exploration](#3-data-loading)
4. [Preprocessing Pipeline](#4-preprocessing)
   - 4.1 [Hospital History Features](#4.1-hospital-history)
   - 4.2 [ICD9 Diagnosis Features](#4.2-icd9-features)
   - 4.3 [Leakage Column Removal](#4.3-leakage-removal)
   - 4.4 [Age Calculation](#4.4-age-calculation)
   - 4.5 [Missing Value Imputation](#4.5-imputation)
   - 4.6 [Categorical Encoding](#4.6-encoding)
   - 4.7 [Medical Feature Engineering](#4.7-feature-engineering)
   - 4.8 [Feature Scaling](#4.8-scaling)
5. [Model Selection & Baseline](#5-model-selection)
6. [Hyperparameter Tuning](#6-hyperparameter-tuning)
7. [Final Model & Evaluation](#7-final-model)
8. [Predictions Generation](#8-predictions)
9. [Conclusion](#9-conclusion)

---

<a id='1-introduction'></a>
## 1. Introduction & Problem Statement

### Objective
Predict the probability of in-hospital mortality (`HOSPITAL_EXPIRE_FLAG`) for ICU patients using the MIMIC-III dataset.

### Dataset Overview
- **Source:** MIMIC-III (Medical Information Mart for Intensive Care III)
- **Observations:** 20,885 ICU stays (training) + 5,221 (test)
- **Target:** Binary classification (0 = survived, 1 = died in hospital)
- **Class Imbalance:** ~11.2% mortality rate (approximately 8:1 ratio)

### Approach
1. **Feature Engineering:** Extract clinically meaningful features from vital signs, diagnoses, and patient history
2. **Handle Class Imbalance:** Use stratified cross-validation and appropriate class weights
3. **Model Selection:** Compare baseline models and ensemble methods
4. **Evaluation Metric:** ROC-AUC (appropriate for imbalanced binary classification)

### Key Challenges
- **Data Leakage:** Several columns reveal the outcome (e.g., DEATHTIME, DOD) and must be removed
- **Missing Data:** ~10-12% missing values in vital sign measurements
- **High Cardinality:** ICD9 diagnosis codes have 1,800+ unique values
- **Age Encoding:** MIMIC-III shifts birth dates for elderly patients (>89 years) for privacy

<a id='2-setup'></a>
## 2. Setup & Imports

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================

# Core libraries
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Model Selection & Evaluation
from sklearn.model_selection import (
    train_test_split, 
    StratifiedKFold, 
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.metrics import (
    roc_auc_score, 
    classification_report, 
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ All imports successful")
print(f"\nLibrary versions:")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")
import sklearn
print(f"  Scikit-learn: {sklearn.__version__}")

✓ All imports successful

Library versions:
  NumPy: 2.3.4
  Pandas: 2.3.3
  Scikit-learn: 1.7.2


<a id='3-data-loading'></a>
## 3. Data Loading & Exploration

In [2]:
# =============================================================================
# DATA LOADING
# =============================================================================

# Define data path (adjust as needed)
data_path = Path("../data/")

# Load main datasets
train_raw = pd.read_csv(data_path / "mimic_train_HEF.csv", low_memory=False)
test_raw = pd.read_csv(data_path / "mimic_test_HEF.csv", low_memory=False)

# Load diagnoses data (for ICD9 feature engineering)
diagnoses_raw = pd.read_csv(data_path / "extra_data" / "MIMIC_diagnoses.csv")

# Create working copies
train = train_raw.copy()
test = test_raw.copy()
diagnoses = diagnoses_raw.copy()

# Standardize diagnoses column names to uppercase
diagnoses.columns = diagnoses.columns.str.upper()

print("="*70)
print("DATA LOADED SUCCESSFULLY")
print("="*70)
print(f"\nDataset shapes:")
print(f"  Training set:   {train.shape[0]:,} samples × {train.shape[1]} features")
print(f"  Test set:       {test.shape[0]:,} samples × {test.shape[1]} features")
print(f"  Diagnoses:      {diagnoses.shape[0]:,} records")

DATA LOADED SUCCESSFULLY

Dataset shapes:
  Training set:   20,885 samples × 44 features
  Test set:       5,221 samples × 39 features
  Diagnoses:      651,047 records


In [3]:
# =============================================================================
# INITIAL DATA EXPLORATION
# =============================================================================

print("="*70)
print("DATA STRUCTURE OVERVIEW")
print("="*70)

# Check for duplicates
train_dupes = train['icustay_id'].duplicated().sum()
test_dupes = test['icustay_id'].duplicated().sum()

print(f"\n1. Duplicate Check:")
print(f"   Train duplicates: {train_dupes}")
print(f"   Test duplicates:  {test_dupes}")
print(f"   Status: {'✓ No duplicates' if train_dupes == 0 and test_dupes == 0 else '⚠ Duplicates found!'}")

# ID structure
print(f"\n2. ID Structure:")
print(f"   Unique patients (subject_id):    {train['subject_id'].nunique():,}")
print(f"   Unique admissions (hadm_id):     {train['hadm_id'].nunique():,}")
print(f"   Unique ICU stays (icustay_id):   {train['icustay_id'].nunique():,}")

# Patient visit statistics
visits_per_patient = train.groupby('subject_id').size()
print(f"\n3. Patient Visit Statistics:")
print(f"   Mean ICU stays per patient:  {visits_per_patient.mean():.2f}")
print(f"   Median:                      {visits_per_patient.median():.0f}")
print(f"   Max:                         {visits_per_patient.max():.0f}")
print(f"   Patients with multiple visits: {(visits_per_patient > 1).sum():,} ({(visits_per_patient > 1).mean()*100:.1f}%)")

# Target distribution
print(f"\n4. Target Variable Distribution:")
mortality_rate = train['HOSPITAL_EXPIRE_FLAG'].mean()
print(f"   Mortality rate: {mortality_rate:.3f} ({mortality_rate*100:.1f}%)")
print(f"   Survived (0):   {(train['HOSPITAL_EXPIRE_FLAG'] == 0).sum():,}")
print(f"   Died (1):       {(train['HOSPITAL_EXPIRE_FLAG'] == 1).sum():,}")
print(f"   Class ratio:    {(1-mortality_rate)/mortality_rate:.1f}:1")

DATA STRUCTURE OVERVIEW

1. Duplicate Check:
   Train duplicates: 0
   Test duplicates:  0
   Status: ✓ No duplicates

2. ID Structure:
   Unique patients (subject_id):    16,317
   Unique admissions (hadm_id):     19,749
   Unique ICU stays (icustay_id):   20,885

3. Patient Visit Statistics:
   Mean ICU stays per patient:  1.28
   Median:                      1
   Max:                         25
   Patients with multiple visits: 2,940 (18.0%)

4. Target Variable Distribution:
   Mortality rate: 0.112 (11.2%)
   Survived (0):   18,540
   Died (1):       2,345
   Class ratio:    7.9:1


In [4]:
# =============================================================================
# COLUMN ANALYSIS
# =============================================================================

print("="*70)
print("COLUMN ANALYSIS")
print("="*70)

# Identify column types
print(f"\nTraining set columns ({train.shape[1]} total):")
print(f"\n{'Column':<25} {'Type':<15} {'Non-Null':<12} {'Unique':<10} {'Sample Values'}")
print("-"*90)

for col in train.columns:
    dtype = str(train[col].dtype)
    non_null = train[col].notna().sum()
    n_unique = train[col].nunique()
    
    # Get sample values
    if train[col].dtype == 'object':
        samples = train[col].dropna().unique()[:2]
        sample_str = ', '.join([str(s)[:15] for s in samples])
    else:
        sample_str = f"range: [{train[col].min():.1f}, {train[col].max():.1f}]"
    
    print(f"{col:<25} {dtype:<15} {non_null:<12} {n_unique:<10} {sample_str[:30]}")

# Columns only in train (potential leakage)
train_only_cols = set(train.columns) - set(test.columns)
print(f"\n⚠ Columns only in training set (potential leakage):")
for col in train_only_cols:
    print(f"   - {col}")

COLUMN ANALYSIS

Training set columns (44 total):

Column                    Type            Non-Null     Unique     Sample Values
------------------------------------------------------------------------------------------
HOSPITAL_EXPIRE_FLAG      int64           20885        2          range: [0.0, 1.0]
subject_id                int64           20885        16317      range: [23.0, 99999.0]
hadm_id                   int64           20885        19749      range: [100001.0, 199999.0]
icustay_id                int64           20885        20885      range: [200001.0, 299998.0]
HeartRate_Min             float64         18698        131        range: [2.0, 141.0]
HeartRate_Max             float64         18698        164        range: [39.0, 280.0]
HeartRate_Mean            float64         18698        14091      range: [34.7, 163.9]
SysBP_Min                 float64         18677        154        range: [5.0, 181.0]
SysBP_Max                 float64         18677        190        range

<a id='4-preprocessing'></a>
## 4. Preprocessing Pipeline

The preprocessing pipeline consists of several carefully ordered steps:

1. **Hospital History Features** - Extract information about patient's previous visits
2. **ICD9 Diagnosis Features** - Create features from diagnosis codes
3. **Leakage Column Removal** - Remove columns that reveal the outcome
4. **Age Calculation** - Convert DOB to age (handling MIMIC-III privacy shifts)
5. **Missing Value Imputation** - Handle missing values appropriately
6. **Categorical Encoding** - Encode categorical variables
7. **Medical Feature Engineering** - Create clinically meaningful features
8. **Feature Scaling** - Standardize continuous features only

<a id='4.1-hospital-history'></a>
### 4.1 Hospital History Features

**Rationale:** Patients with multiple ICU visits may have different mortality risk profiles. We create features to capture this history.

**Features created:**
- `n_previous_icu_stays`: Count of previous ICU visits for this patient
- `is_first_icu_visit`: Binary flag for first-time ICU patients
- `is_frequent_flyer`: Binary flag for patients with 3+ visits

In [5]:
# =============================================================================
# 4.1 HOSPITAL HISTORY FEATURES
# =============================================================================

def create_hospital_history_features(df, df_name="dataset"):
    """
    Create features based on patient's ICU visit history.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dataset with subject_id and ADMITTIME columns
    df_name : str
        Name for logging purposes
        
    Returns:
    --------
    pd.DataFrame
        Dataset with new history features added
    """
    df = df.copy()
    
    # Sort by patient and admission time to ensure correct ordering
    if 'ADMITTIME' in df.columns:
        df['ADMITTIME'] = pd.to_datetime(df['ADMITTIME'], errors='coerce')
        df = df.sort_values(['subject_id', 'ADMITTIME'])
    else:
        df = df.sort_values(['subject_id', 'hadm_id', 'icustay_id'])
    
    # Feature 1: Count of previous ICU stays for this patient
    df['n_previous_icu_stays'] = df.groupby('subject_id').cumcount()
    
    # Feature 2: Is this the patient's first ICU visit?
    df['is_first_icu_visit'] = (df['n_previous_icu_stays'] == 0).astype(int)
    
    # Feature 3: Is this patient a "frequent flyer" (3+ total visits)?
    total_visits = df.groupby('subject_id').size()
    frequent_patients = total_visits[total_visits >= 3].index
    df['is_frequent_flyer'] = df['subject_id'].isin(frequent_patients).astype(int)
    
    return df

# Apply to train and test
train = create_hospital_history_features(train, "train")
test = create_hospital_history_features(test, "test")

# CRITICAL: Save test IDs in their current order (after sorting)
# This ensures predictions align with the correct patients
test_ids = test['icustay_id'].copy()

# Validation
print("="*70)
print("4.1 HOSPITAL HISTORY FEATURES")
print("="*70)

print(f"\nFeatures created:")
for feat in ['n_previous_icu_stays', 'is_first_icu_visit', 'is_frequent_flyer']:
    print(f"\n  {feat}:")
    print(f"    Train - Min: {train[feat].min()}, Max: {train[feat].max()}, Mean: {train[feat].mean():.3f}")
    print(f"    Test  - Min: {test[feat].min()}, Max: {test[feat].max()}, Mean: {test[feat].mean():.3f}")

print(f"\n✓ Saved {len(test_ids)} test IDs in correct order")

4.1 HOSPITAL HISTORY FEATURES

Features created:

  n_previous_icu_stays:
    Train - Min: 0, Max: 24, Mean: 0.396
    Test  - Min: 0, Max: 4, Mean: 0.088

  is_first_icu_visit:
    Train - Min: 0, Max: 1, Mean: 0.781
    Test  - Min: 0, Max: 1, Mean: 0.928

  is_frequent_flyer:
    Train - Min: 0, Max: 1, Mean: 0.162
    Test  - Min: 0, Max: 1, Mean: 0.033

✓ Saved 5221 test IDs in correct order


<a id='4.2-icd9-features'></a>
### 4.2 ICD9 Diagnosis Features

**Rationale:** Diagnosis codes contain rich clinical information. We extract:
- Number of diagnoses (complexity indicator)
- Primary diagnosis category (first 3 digits of ICD9)
- Major disease category
- High-risk condition flags (sepsis, heart failure, etc.)

In [6]:
# =============================================================================
# 4.2 ICD9 DIAGNOSIS FEATURES
# =============================================================================

print("="*70)
print("4.2 ICD9 DIAGNOSIS FEATURES")
print("="*70)

# --- Feature 1: Number of diagnoses per admission ---
n_diagnoses_per_admission = diagnoses.groupby('HADM_ID').size()
train['n_diagnoses'] = train['hadm_id'].map(n_diagnoses_per_admission).fillna(0).astype(int)
test['n_diagnoses'] = test['hadm_id'].map(n_diagnoses_per_admission).fillna(0).astype(int)

print(f"\n1. Number of diagnoses per admission:")
print(f"   Mean: {n_diagnoses_per_admission.mean():.1f}, Median: {n_diagnoses_per_admission.median():.0f}, Max: {n_diagnoses_per_admission.max():.0f}")

# --- Feature 2: Primary diagnosis (SEQ_NUM = 1) ---
primary_diagnoses = diagnoses[diagnoses['SEQ_NUM'] == 1][['HADM_ID', 'ICD9_CODE']].set_index('HADM_ID')['ICD9_CODE']
train['primary_diagnosis_raw'] = train['hadm_id'].map(primary_diagnoses)
test['primary_diagnosis_raw'] = test['hadm_id'].map(primary_diagnoses)

# --- Feature 3: ICD9 category (first 3 characters) ---
def extract_icd9_category(code):
    """Extract first 3 characters from ICD9 code for category grouping."""
    if pd.isna(code):
        return 'UNKNOWN'
    code_str = str(code).strip().replace('.', '').replace(' ', '')
    return code_str[:3] if len(code_str) >= 3 else (code_str if len(code_str) > 0 else 'UNKNOWN')

train['primary_diag_cat'] = train['primary_diagnosis_raw'].apply(extract_icd9_category)
test['primary_diag_cat'] = test['primary_diagnosis_raw'].apply(extract_icd9_category)

print(f"\n2. Primary diagnosis categories: {train['primary_diag_cat'].nunique()} unique")

# --- Feature 4: Major disease category ---
def get_disease_category(code):
    """Map ICD9 code to major disease category based on first digit."""
    if pd.isna(code):
        return 'UNKNOWN'
    
    code_str = str(code).strip().replace('.', '').replace(' ', '')
    if len(code_str) == 0:
        return 'UNKNOWN'
    
    first_char = code_str[0].upper()
    
    # ICD9 structure mapping
    category_map = {
        '0': 'INFECTIOUS', '1': 'INFECTIOUS',
        '2': 'NEOPLASM',
        '3': 'ENDOCRINE',
        '4': 'BLOOD',
        '5': 'MENTAL',
        '6': 'NERVOUS', '7': 'NERVOUS',
        '8': 'CIRCULATORY',
        '9': 'RESPIRATORY',
        'V': 'V_CODE',
        'E': 'E_CODE'
    }
    return category_map.get(first_char, 'OTHER')

train['disease_category'] = train['primary_diagnosis_raw'].apply(get_disease_category)
test['disease_category'] = test['primary_diagnosis_raw'].apply(get_disease_category)

print(f"\n3. Major disease categories:")
for cat, count in train['disease_category'].value_counts().head(5).items():
    print(f"   {cat}: {count} ({count/len(train)*100:.1f}%)")

4.2 ICD9 DIAGNOSIS FEATURES

1. Number of diagnoses per admission:
   Mean: 11.0, Median: 9, Max: 39

2. Primary diagnosis categories: 530 unique

3. Major disease categories:
   BLOOD: 7507 (35.9%)
   MENTAL: 3912 (18.7%)
   INFECTIOUS: 3208 (15.4%)
   CIRCULATORY: 1795 (8.6%)
   RESPIRATORY: 1578 (7.6%)


In [7]:
# --- Feature 5: High-risk condition flags ---
# Build efficient lookup: hadm_id -> set of all ICD9 codes
all_diagnoses_per_admission = diagnoses.groupby('HADM_ID')['ICD9_CODE'].apply(
    lambda x: set(str(code).replace('.', '').replace(' ', '') for code in x)
)

def check_condition_presence(hadm_id, code_patterns):
    """Check if any diagnosis code matches the given patterns."""
    if hadm_id not in all_diagnoses_per_admission.index:
        return 0
    codes = all_diagnoses_per_admission[hadm_id]
    return 1 if any(code.startswith(pattern) for code in codes for pattern in code_patterns) else 0

# Define high-risk conditions with their ICD9 code patterns
conditions = {
    'has_sepsis': ['99591', '99592', '78552'],           # Sepsis/severe sepsis
    'has_heart_failure': ['428'],                        # Heart failure
    'has_respiratory_failure': ['518'],                  # Respiratory failure
    'has_aki': ['584'],                                  # Acute kidney injury
    'has_diabetes': ['250'],                             # Diabetes mellitus
    'has_copd': ['491', '492', '496'],                   # COPD
    'has_pneumonia': ['480', '481', '482', '483', '484', '485', '486']  # Pneumonia
}

print(f"\n4. High-risk condition flags:")
for condition_name, patterns in conditions.items():
    train[condition_name] = train['hadm_id'].apply(lambda x: check_condition_presence(x, patterns))
    test[condition_name] = test['hadm_id'].apply(lambda x: check_condition_presence(x, patterns))
    
    count = train[condition_name].sum()
    print(f"   {condition_name}: {count} ({count/len(train)*100:.1f}%)")

print(f"\n✓ ICD9 features created successfully")


4. High-risk condition flags:
   has_sepsis: 2805 (13.4%)
   has_heart_failure: 5463 (26.2%)
   has_respiratory_failure: 6116 (29.3%)
   has_aki: 5842 (28.0%)
   has_diabetes: 6125 (29.3%)
   has_copd: 2749 (13.2%)
   has_pneumonia: 3177 (15.2%)

✓ ICD9 features created successfully


<a id='4.3-leakage-removal'></a>
### 4.3 Leakage Column Removal

**Rationale:** Several columns in the dataset would leak information about the outcome:
- `DEATHTIME`, `DOD`: Directly reveal if patient died
- `DISCHTIME`: Reveals discharge timing (related to outcome)
- `LOS`: Length of stay correlates with outcome
- ID columns: No predictive value, already used for feature engineering

In [8]:
# =============================================================================
# 4.3 LEAKAGE COLUMN REMOVAL
# =============================================================================

print("="*70)
print("4.3 LEAKAGE COLUMN REMOVAL")
print("="*70)

# Define columns to drop
leakage_columns = [
    'DISCHTIME',      # Discharge time - only known after outcome
    'DEATHTIME',      # Death time - IS the target
    'DOD',            # Date of death - IS the target
    'LOS',            # Length of stay - correlated with outcome
    'Diff',           # Time difference - likely leakage
    'ADMITTIME',      # Already used for history features
]

id_columns = [
    'icustay_id',     # Already saved as test_ids
    'subject_id',     # Used for history features
    'hadm_id',        # Used for diagnosis matching
]

other_columns = [
    'primary_diagnosis_raw'  # Will use encoded versions only
]

all_columns_to_drop = leakage_columns + id_columns + other_columns

print(f"\nColumns to remove:")
for col in all_columns_to_drop:
    train_has = "✓" if col in train.columns else "✗"
    test_has = "✓" if col in test.columns else "✗"
    print(f"  {col:<25} Train:{train_has}  Test:{test_has}")

# Drop columns
train = train.drop(columns=[c for c in all_columns_to_drop if c in train.columns], errors='ignore')
test = test.drop(columns=[c for c in all_columns_to_drop if c in test.columns], errors='ignore')

# Separate target variable
y = train['HOSPITAL_EXPIRE_FLAG'].copy()
X = train.drop('HOSPITAL_EXPIRE_FLAG', axis=1)
X_test = test.copy()

print(f"\nAfter removal:")
print(f"  X shape:      {X.shape}")
print(f"  y shape:      {y.shape}")
print(f"  X_test shape: {X_test.shape}")

# Verify target
print(f"\nTarget validation:")
print(f"  Mortality rate: {y.mean():.3f} ({y.sum()}/{len(y)})")
print(f"  ✓ Matches expected ~11.2%" if abs(y.mean() - 0.112) < 0.01 else "⚠ Unexpected mortality rate")

4.3 LEAKAGE COLUMN REMOVAL

Columns to remove:
  DISCHTIME                 Train:✓  Test:✗
  DEATHTIME                 Train:✓  Test:✗
  DOD                       Train:✓  Test:✗
  LOS                       Train:✓  Test:✗
  Diff                      Train:✓  Test:✓
  ADMITTIME                 Train:✓  Test:✓
  icustay_id                Train:✓  Test:✓
  subject_id                Train:✓  Test:✓
  hadm_id                   Train:✓  Test:✓
  primary_diagnosis_raw     Train:✓  Test:✓

After removal:
  X shape:      (20885, 47)
  y shape:      (20885,)
  X_test shape: (5221, 47)

Target validation:
  Mortality rate: 0.112 (2345/20885)
  ✓ Matches expected ~11.2%


<a id='4.4-age-calculation'></a>
### 4.4 Age Calculation

**Rationale:** MIMIC-III shifts birth dates for patients >89 years old for privacy protection. This results in ages appearing as >200 or negative values. We calculate age from DOB and ADMITTIME, then handle invalid values.

In [9]:
# =============================================================================
# 4.4 AGE CALCULATION
# =============================================================================

print("="*70)
print("4.4 AGE CALCULATION")
print("="*70)

if 'DOB' in X.columns:
    # Reload original data for ADMITTIME (we dropped it earlier)
    train_original = pd.read_csv(data_path / 'mimic_train_HEF.csv')
    test_original = pd.read_csv(data_path / 'mimic_test_HEF.csv')
    
    # Convert to datetime
    dob_train = pd.to_datetime(X['DOB'], errors='coerce')
    dob_test = pd.to_datetime(X_test['DOB'], errors='coerce')
    admit_train = pd.to_datetime(train_original['ADMITTIME'], errors='coerce')
    admit_test = pd.to_datetime(test_original['ADMITTIME'], errors='coerce')
    
    # Calculate age in years
    def calculate_age(admit_time, dob):
        """Calculate age in years from admission time and DOB."""
        if pd.isna(admit_time) or pd.isna(dob):
            return np.nan
        try:
            return (admit_time - dob).days / 365.25
        except:
            return np.nan
    
    X['age'] = [calculate_age(a, d) for a, d in zip(admit_train, dob_train)]
    X_test['age'] = [calculate_age(a, d) for a, d in zip(admit_test, dob_test)]
    
    print(f"\nAge distribution (before cleaning):")
    print(f"  Min: {X['age'].min():.1f}, Max: {X['age'].max():.1f}")
    print(f"  Invalid ages (< 0 or > 120): {((X['age'] < 0) | (X['age'] > 120)).sum()}")
    
    # Handle invalid ages (MIMIC-III privacy shifts)
    invalid_train = (X['age'] < 0) | (X['age'] > 120)
    invalid_test = (X_test['age'] < 0) | (X_test['age'] > 120)
    
    X.loc[invalid_train, 'age'] = np.nan
    X_test.loc[invalid_test, 'age'] = np.nan
    
    # Impute with median
    age_median = X['age'].median()
    X['age'].fillna(age_median, inplace=True)
    X_test['age'].fillna(age_median, inplace=True)
    
    # Drop DOB column
    X = X.drop('DOB', axis=1)
    X_test = X_test.drop('DOB', axis=1)
    
    print(f"\nAge distribution (after cleaning):")
    print(f"  Range: {X['age'].min():.1f} - {X['age'].max():.1f} years")
    print(f"  Mean: {X['age'].mean():.1f}, Median: {X['age'].median():.1f}")
    print(f"  Imputed with median: {age_median:.1f}")
    
print(f"\n✓ Age calculation complete")

4.4 AGE CALCULATION

Age distribution (before cleaning):
  Min: -71.9, Max: 292.0
  Invalid ages (< 0 or > 120): 4340

Age distribution (after cleaning):
  Range: 0.0 - 120.0 years
  Mean: 61.4, Median: 62.0
  Imputed with median: 62.0

✓ Age calculation complete


<a id='4.5-imputation'></a>
### 4.5 Missing Value Imputation

**Strategy:**
- **Numeric features:** Median imputation (robust to outliers)
- **Categorical features:** Mode imputation (most frequent value)

In [10]:
# =============================================================================
# 4.5 MISSING VALUE IMPUTATION
# =============================================================================

print("="*70)
print("4.5 MISSING VALUE IMPUTATION")
print("="*70)

# Identify feature types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\nFeature types:")
print(f"  Numeric features:     {len(numeric_features)}")
print(f"  Categorical features: {len(categorical_features)}")

# Analyze missing values
print(f"\nMissing values (numeric):")
missing_numeric = X[numeric_features].isnull().sum()
missing_numeric = missing_numeric[missing_numeric > 0].sort_values(ascending=False)

if len(missing_numeric) > 0:
    for feat, count in missing_numeric.head(10).items():
        print(f"  {feat:<30} {count:>6} ({count/len(X)*100:>5.1f}%)")
else:
    print("  None")

# Impute numeric features with median
if len(numeric_features) > 0:
    numeric_imputer = SimpleImputer(strategy='median')
    X[numeric_features] = numeric_imputer.fit_transform(X[numeric_features])
    X_test[numeric_features] = numeric_imputer.transform(X_test[numeric_features])
    print(f"\n✓ Imputed {len(numeric_features)} numeric features with median")

# Impute categorical features with mode
if len(categorical_features) > 0:
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    X[categorical_features] = categorical_imputer.fit_transform(X[categorical_features])
    X_test[categorical_features] = categorical_imputer.transform(X_test[categorical_features])
    print(f"✓ Imputed {len(categorical_features)} categorical features with mode")

# Verify no missing values remain
assert X.isnull().sum().sum() == 0, "Missing values still present in train!"
assert X_test.isnull().sum().sum() == 0, "Missing values still present in test!"
print(f"\n✓ Validation passed: No missing values remain")

4.5 MISSING VALUE IMPUTATION

Feature types:
  Numeric features:     36
  Categorical features: 11

Missing values (numeric):
  TempC_Max                        2497 ( 12.0%)
  TempC_Mean                       2497 ( 12.0%)
  TempC_Min                        2497 ( 12.0%)
  DiasBP_Min                       2209 ( 10.6%)
  DiasBP_Mean                      2209 ( 10.6%)
  DiasBP_Max                       2209 ( 10.6%)
  SysBP_Min                        2208 ( 10.6%)
  SysBP_Mean                       2208 ( 10.6%)
  SysBP_Max                        2208 ( 10.6%)
  SpO2_Max                         2203 ( 10.5%)

✓ Imputed 36 numeric features with median
✓ Imputed 11 categorical features with mode

✓ Validation passed: No missing values remain


<a id='4.6-encoding'></a>
### 4.6 Categorical Encoding

**Strategy:**
- **High cardinality (ICD9, diagnosis categories):** Target encoding
- **Medium cardinality (ethnicity, religion):** Group similar categories, then one-hot encode
- **Low cardinality (gender, admission type):** One-hot encoding

In [11]:
# =============================================================================
# 4.6 CATEGORICAL ENCODING
# =============================================================================

print("="*70)
print("4.6 CATEGORICAL ENCODING")
print("="*70)

# --- Target encoding for high-cardinality features ---
print(f"\n1. Target encoding (high cardinality):")

global_mean = y.mean()

# ICD9_diagnosis -> extract category and target encode
if 'ICD9_diagnosis' in X.columns:
    X['ICD9_cat'] = X['ICD9_diagnosis'].apply(extract_icd9_category)
    X_test['ICD9_cat'] = X_test['ICD9_diagnosis'].apply(extract_icd9_category)
    
    encoding_map = y.groupby(X['ICD9_cat']).mean().to_dict()
    X['ICD9_encoded'] = X['ICD9_cat'].map(encoding_map)
    X_test['ICD9_encoded'] = X_test['ICD9_cat'].map(encoding_map).fillna(global_mean)
    
    X = X.drop(['ICD9_diagnosis', 'ICD9_cat'], axis=1)
    X_test = X_test.drop(['ICD9_diagnosis', 'ICD9_cat'], axis=1)
    categorical_features.remove('ICD9_diagnosis')
    print(f"   ✓ ICD9_diagnosis target encoded")

# primary_diag_cat -> target encode
if 'primary_diag_cat' in X.columns:
    encoding_map = y.groupby(X['primary_diag_cat']).mean().to_dict()
    X['primary_diag_encoded'] = X['primary_diag_cat'].map(encoding_map)
    X_test['primary_diag_encoded'] = X_test['primary_diag_cat'].map(encoding_map).fillna(global_mean)
    
    X = X.drop('primary_diag_cat', axis=1)
    X_test = X_test.drop('primary_diag_cat', axis=1)
    categorical_features.remove('primary_diag_cat')
    print(f"   ✓ primary_diag_cat target encoded")

# --- Drop free text DIAGNOSIS column ---
print(f"\n2. Dropping high-cardinality text:")
if 'DIAGNOSIS' in categorical_features:
    X = X.drop('DIAGNOSIS', axis=1)
    X_test = X_test.drop('DIAGNOSIS', axis=1)
    categorical_features.remove('DIAGNOSIS')
    print(f"   ✓ DIAGNOSIS dropped (free text, {train_raw['DIAGNOSIS'].nunique()} unique)")

4.6 CATEGORICAL ENCODING

1. Target encoding (high cardinality):
   ✓ ICD9_diagnosis target encoded
   ✓ primary_diag_cat target encoded

2. Dropping high-cardinality text:
   ✓ DIAGNOSIS dropped (free text, 6193 unique)


In [12]:
# --- Group medium-cardinality categories ---
print(f"\n3. Grouping categories:")

# ETHNICITY grouping
if 'ETHNICITY' in categorical_features:
    def group_ethnicity(ethnicity):
        if pd.isna(ethnicity):
            return 'UNKNOWN'
        ethnicity = str(ethnicity).upper()
        if 'WHITE' in ethnicity:
            return 'WHITE'
        elif 'BLACK' in ethnicity or 'AFRICAN' in ethnicity:
            return 'BLACK'
        elif 'HISPANIC' in ethnicity or 'LATINO' in ethnicity:
            return 'HISPANIC'
        elif 'ASIAN' in ethnicity:
            return 'ASIAN'
        elif any(x in ethnicity for x in ['UNKNOWN', 'UNABLE', 'DECLINED', 'NOT SPECIFIED']):
            return 'UNKNOWN'
        else:
            return 'OTHER'
    
    X['ETHNICITY'] = X['ETHNICITY'].apply(group_ethnicity)
    X_test['ETHNICITY'] = X_test['ETHNICITY'].apply(group_ethnicity)
    print(f"   ✓ ETHNICITY: {X['ETHNICITY'].nunique()} groups")

# RELIGION grouping
if 'RELIGION' in categorical_features:
    def group_religion(religion):
        if pd.isna(religion):
            return 'UNKNOWN'
        religion = str(religion).upper()
        if 'CATHOLIC' in religion:
            return 'CATHOLIC'
        elif any(x in religion for x in ['PROTESTANT', 'EPISCOPALIAN', 'QUAKER']):
            return 'PROTESTANT'
        elif 'JEWISH' in religion:
            return 'JEWISH'
        elif any(x in religion for x in ['UNOBTAINABLE', 'NOT SPECIFIED', 'UNKNOWN']):
            return 'UNKNOWN'
        else:
            return 'OTHER'
    
    X['RELIGION'] = X['RELIGION'].apply(group_religion)
    X_test['RELIGION'] = X_test['RELIGION'].apply(group_religion)
    print(f"   ✓ RELIGION: {X['RELIGION'].nunique()} groups")

# MARITAL_STATUS grouping
if 'MARITAL_STATUS' in categorical_features:
    def group_marital_status(status):
        if pd.isna(status):
            return 'UNKNOWN'
        status = str(status).upper()
        if 'MARRIED' in status or 'LIFE PARTNER' in status:
            return 'MARRIED'
        elif 'SINGLE' in status:
            return 'SINGLE'
        elif 'WIDOWED' in status:
            return 'WIDOWED'
        elif 'DIVORCED' in status or 'SEPARATED' in status:
            return 'DIVORCED_SEPARATED'
        else:
            return 'UNKNOWN'
    
    X['MARITAL_STATUS'] = X['MARITAL_STATUS'].apply(group_marital_status)
    X_test['MARITAL_STATUS'] = X_test['MARITAL_STATUS'].apply(group_marital_status)
    print(f"   ✓ MARITAL_STATUS: {X['MARITAL_STATUS'].nunique()} groups")


3. Grouping categories:
   ✓ ETHNICITY: 6 groups
   ✓ RELIGION: 5 groups
   ✓ MARITAL_STATUS: 5 groups


In [13]:
# --- One-hot encode remaining categorical features ---
print(f"\n4. One-hot encoding:")

remaining_categorical = [col for col in categorical_features if col in X.columns]
if 'disease_category' in X.columns and 'disease_category' not in remaining_categorical:
    remaining_categorical.append('disease_category')

print(f"   Features to encode: {remaining_categorical}")

if len(remaining_categorical) > 0:
    # Combine train and test for consistent encoding
    X_combined = pd.concat([X, X_test], keys=['train', 'test'])
    
    # One-hot encode
    X_encoded = pd.get_dummies(
        X_combined, 
        columns=remaining_categorical, 
        drop_first=True,
        dtype=int
    )
    
    # Split back
    X = X_encoded.xs('train')
    X_test = X_encoded.xs('test')

# Validation
object_cols = X.select_dtypes(include=['object']).columns.tolist()
assert len(object_cols) == 0, f"Object columns remain: {object_cols}"
assert list(X.columns) == list(X_test.columns), "Column mismatch!"

print(f"\n✓ All categorical features encoded")
print(f"   Final feature count: {X.shape[1]}")


4. One-hot encoding:
   Features to encode: ['GENDER', 'ADMISSION_TYPE', 'INSURANCE', 'RELIGION', 'MARITAL_STATUS', 'ETHNICITY', 'FIRST_CAREUNIT', 'disease_category']

✓ All categorical features encoded
   Final feature count: 70


<a id='4.7-feature-engineering'></a>
### 4.7 Medical Feature Engineering

**Rationale:** Create clinically meaningful features that capture patient acuity:
- **Shock indices:** Heart rate / blood pressure ratios (hemodynamic instability)
- **Vital sign ranges:** Variability indicates instability
- **Clinical thresholds:** Binary flags for concerning values (hypoxemia, fever, etc.)
- **Composite severity score:** Aggregated risk indicator

In [14]:
# =============================================================================
# 4.7 MEDICAL FEATURE ENGINEERING
# =============================================================================

print("="*70)
print("4.7 MEDICAL FEATURE ENGINEERING")
print("="*70)

original_features = X.shape[1]

# --- Blood Pressure Features ---
print(f"\n1. Blood Pressure Features:")

if all(col in X.columns for col in ['SysBP_Mean', 'DiasBP_Mean']):
    X['PulsePressure'] = X['SysBP_Mean'] - X['DiasBP_Mean']
    X_test['PulsePressure'] = X_test['SysBP_Mean'] - X_test['DiasBP_Mean']
    print(f"   ✓ Pulse pressure (SysBP - DiasBP)")

if all(col in X.columns for col in ['SysBP_Min', 'SysBP_Max']):
    X['SysBP_Range'] = X['SysBP_Max'] - X['SysBP_Min']
    X_test['SysBP_Range'] = X_test['SysBP_Max'] - X_test['SysBP_Min']
    print(f"   ✓ Systolic BP range (variability)")

# --- Shock Indices (critical for ICU mortality prediction) ---
print(f"\n2. Shock Indices:")

if all(col in X.columns for col in ['HeartRate_Mean', 'SysBP_Mean']):
    # Standard Shock Index: HR / SBP (normal < 0.7, elevated > 0.9)
    X['ShockIndex'] = (X['HeartRate_Mean'] / (X['SysBP_Mean'] + 1)).clip(0, 3)
    X_test['ShockIndex'] = (X_test['HeartRate_Mean'] / (X_test['SysBP_Mean'] + 1)).clip(0, 3)
    print(f"   ✓ Shock Index (HR/SBP, clipped 0-3)")

if all(col in X.columns for col in ['HeartRate_Mean', 'MeanBP_Mean']):
    # Modified Shock Index: HR / MAP
    X['ModifiedShockIndex'] = (X['HeartRate_Mean'] / (X['MeanBP_Mean'] + 1)).clip(0, 3)
    X_test['ModifiedShockIndex'] = (X_test['HeartRate_Mean'] / (X_test['MeanBP_Mean'] + 1)).clip(0, 3)
    print(f"   ✓ Modified Shock Index (HR/MAP, clipped 0-3)")

# --- Clinical Threshold Indicators ---
print(f"\n3. Clinical Indicators:")

# Respiratory
if 'SpO2_Min' in X.columns:
    X['Hypoxemia'] = (X['SpO2_Min'] < 90).astype(int)
    X_test['Hypoxemia'] = (X_test['SpO2_Min'] < 90).astype(int)
    print(f"   ✓ Hypoxemia (SpO2 < 90%)")

if 'RespRate_Mean' in X.columns:
    X['RespRate_Abnormal'] = ((X['RespRate_Mean'] < 12) | (X['RespRate_Mean'] > 20)).astype(int)
    X_test['RespRate_Abnormal'] = ((X_test['RespRate_Mean'] < 12) | (X_test['RespRate_Mean'] > 20)).astype(int)
    print(f"   ✓ Abnormal respiratory rate (<12 or >20)")

# Temperature
if 'TempC_Max' in X.columns:
    X['Fever'] = (X['TempC_Max'] > 38).astype(int)
    X_test['Fever'] = (X_test['TempC_Max'] > 38).astype(int)
    print(f"   ✓ Fever (temp > 38°C)")

if 'TempC_Min' in X.columns:
    X['Hypothermia'] = (X['TempC_Min'] < 36).astype(int)
    X_test['Hypothermia'] = (X_test['TempC_Min'] < 36).astype(int)
    print(f"   ✓ Hypothermia (temp < 36°C)")

if all(col in X.columns for col in ['TempC_Min', 'TempC_Max']):
    X['Temp_Range'] = X['TempC_Max'] - X['TempC_Min']
    X_test['Temp_Range'] = X_test['TempC_Max'] - X_test['TempC_Min']
    print(f"   ✓ Temperature range")

# Glucose
if 'Glucose_Max' in X.columns:
    X['Hyperglycemia'] = (X['Glucose_Max'] > 180).astype(int)
    X_test['Hyperglycemia'] = (X_test['Glucose_Max'] > 180).astype(int)
    print(f"   ✓ Hyperglycemia (glucose > 180)")

if 'Glucose_Min' in X.columns:
    X['Hypoglycemia'] = (X['Glucose_Min'] < 70).astype(int)
    X_test['Hypoglycemia'] = (X_test['Glucose_Min'] < 70).astype(int)
    print(f"   ✓ Hypoglycemia (glucose < 70)")

if all(col in X.columns for col in ['Glucose_Min', 'Glucose_Max']):
    X['Glucose_Range'] = X['Glucose_Max'] - X['Glucose_Min']
    X_test['Glucose_Range'] = X_test['Glucose_Max'] - X_test['Glucose_Min']
    print(f"   ✓ Glucose variability")

4.7 MEDICAL FEATURE ENGINEERING

1. Blood Pressure Features:
   ✓ Pulse pressure (SysBP - DiasBP)
   ✓ Systolic BP range (variability)

2. Shock Indices:
   ✓ Shock Index (HR/SBP, clipped 0-3)
   ✓ Modified Shock Index (HR/MAP, clipped 0-3)

3. Clinical Indicators:
   ✓ Hypoxemia (SpO2 < 90%)
   ✓ Abnormal respiratory rate (<12 or >20)
   ✓ Fever (temp > 38°C)
   ✓ Hypothermia (temp < 36°C)
   ✓ Temperature range
   ✓ Hyperglycemia (glucose > 180)
   ✓ Hypoglycemia (glucose < 70)
   ✓ Glucose variability


In [15]:
# --- Age-Based Features ---
print(f"\n4. Age Features:")

if 'age' in X.columns:
    # Elderly indicator
    X['Elderly'] = (X['age'] > 65).astype(int)
    X_test['Elderly'] = (X_test['age'] > 65).astype(int)
    print(f"   ✓ Elderly (age > 65)")
    
    # Age squared for non-linear effects
    X['age_squared'] = X['age'] ** 2
    X_test['age_squared'] = X_test['age'] ** 2
    print(f"   ✓ Age squared")
    
    # Age risk groups
    age_bins = [0, 18, 45, 65, 80, 120]
    age_labels = ['pediatric', 'young_adult', 'middle_age', 'elderly', 'very_old']
    
    X['age_group'] = pd.cut(X['age'], bins=age_bins, labels=age_labels)
    X_test['age_group'] = pd.cut(X_test['age'], bins=age_bins, labels=age_labels)
    
    # One-hot encode
    X_combined = pd.concat([X, X_test], keys=['train', 'test'])
    X_encoded = pd.get_dummies(X_combined, columns=['age_group'], drop_first=True, prefix='age', dtype=int)
    X = X_encoded.xs('train')
    X_test = X_encoded.xs('test')
    print(f"   ✓ Age risk groups (one-hot encoded)")

# Heart rate range
if all(col in X.columns for col in ['HeartRate_Min', 'HeartRate_Max']):
    X['HeartRate_Range'] = X['HeartRate_Max'] - X['HeartRate_Min']
    X_test['HeartRate_Range'] = X_test['HeartRate_Max'] - X_test['HeartRate_Min']
    print(f"   ✓ Heart rate range")

# --- Composite Severity Score ---
print(f"\n5. Composite Severity Score:")

severity_cols = ['ShockIndex', 'Hypoxemia', 'RespRate_Abnormal', 'Fever', 'Hypothermia']
existing_cols = [col for col in severity_cols if col in X.columns]

if len(existing_cols) > 0:
    # Score based on elevated shock index + other indicators
    X['Severity_Score'] = 0
    X_test['Severity_Score'] = 0
    
    if 'ShockIndex' in X.columns:
        X['Severity_Score'] += (X['ShockIndex'] > 0.9).astype(int)
        X_test['Severity_Score'] += (X_test['ShockIndex'] > 0.9).astype(int)
    
    for col in ['Hypoxemia', 'RespRate_Abnormal', 'Fever', 'Hypothermia']:
        if col in X.columns:
            X['Severity_Score'] += X[col]
            X_test['Severity_Score'] += X_test[col]
    
    print(f"   ✓ Severity score (0-5): {dict(X['Severity_Score'].value_counts().sort_index())}")

# Summary
new_features = X.shape[1] - original_features
print(f"\n" + "="*70)
print(f"Features added: {new_features}")
print(f"Total features: {X.shape[1]}")


4. Age Features:
   ✓ Elderly (age > 65)
   ✓ Age squared
   ✓ Age risk groups (one-hot encoded)
   ✓ Heart rate range

5. Composite Severity Score:
   ✓ Severity score (0-5): {0: np.int64(6837), 1: np.int64(6834), 2: np.int64(4541), 3: np.int64(2088), 4: np.int64(537), 5: np.int64(48)}

Features added: 20
Total features: 90


<a id='4.8-scaling'></a>
### 4.8 Feature Scaling

**Strategy:** Apply StandardScaler only to continuous features. Binary indicators, count features, and ordinal features should NOT be scaled as their interpretation would be lost.

**Features to scale:** Vital signs, age, engineered continuous features  
**Features NOT to scale:** Binary flags, count features (n_diagnoses), one-hot encoded, severity score

In [16]:
# =============================================================================
# 4.8 FEATURE SCALING
# =============================================================================

print("="*70)
print("4.8 FEATURE SCALING")
print("="*70)

# Identify features to exclude from scaling
all_numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Binary indicators - should remain 0/1
binary_features = [
    'Hypoxemia', 'RespRate_Abnormal', 'Fever', 'Hypothermia',
    'Hyperglycemia', 'Hypoglycemia', 'Elderly',
    'is_first_icu_visit', 'is_frequent_flyer',
    'has_sepsis', 'has_heart_failure', 'has_respiratory_failure',
    'has_aki', 'has_diabetes', 'has_copd', 'has_pneumonia'
]

# Count features - meaningful as integers
count_features = ['n_previous_icu_stays', 'n_diagnoses']

# Ordinal features - meaningful ordering
ordinal_features = ['Severity_Score']

# One-hot encoded features (binary by definition)
one_hot_features = [col for col in X.columns if '_' in col and X[col].nunique() <= 2]

# Combine all features to exclude
exclude_from_scaling = list(set(
    binary_features + count_features + ordinal_features + one_hot_features
))
exclude_from_scaling = [col for col in exclude_from_scaling if col in all_numeric_cols]

# Features to scale
features_to_scale = [col for col in all_numeric_cols if col not in exclude_from_scaling]

print(f"\nScaling strategy:")
print(f"  Features to scale:      {len(features_to_scale)} (continuous)")
print(f"  Features NOT scaled:    {len(exclude_from_scaling)} (binary/count/ordinal)")

print(f"\nSample features to scale: {features_to_scale[:5]}")
print(f"Sample features NOT scaled: {exclude_from_scaling[:5]}")

# Apply StandardScaler
scaler = StandardScaler()
X[features_to_scale] = scaler.fit_transform(X[features_to_scale])
X_test[features_to_scale] = scaler.transform(X_test[features_to_scale])

# Validation
print(f"\nValidation:")
for feat in features_to_scale[:3]:
    print(f"  {feat}: mean={X[feat].mean():.4f}, std={X[feat].std():.4f}")

# Check binary features remain 0/1
for feat in binary_features[:3]:
    if feat in X.columns:
        unique_vals = X[feat].unique()
        assert set(unique_vals).issubset({0, 1, 0.0, 1.0}), f"{feat} was scaled!"
        
print(f"\n✓ Scaling complete and validated")

4.8 FEATURE SCALING

Scaling strategy:
  Features to scale:      35 (continuous)
  Features NOT scaled:    55 (binary/count/ordinal)

Sample features to scale: ['HeartRate_Min', 'HeartRate_Max', 'HeartRate_Mean', 'SysBP_Min', 'SysBP_Max']
Sample features NOT scaled: ['has_diabetes', 'has_copd', 'RELIGION_OTHER', 'MARITAL_STATUS_MARRIED', 'has_aki']

Validation:
  HeartRate_Min: mean=0.0000, std=1.0000
  HeartRate_Max: mean=0.0000, std=1.0000
  HeartRate_Mean: mean=0.0000, std=1.0000

✓ Scaling complete and validated


### 4.9 Final Preprocessing Validation

In [17]:
# =============================================================================
# FINAL PREPROCESSING VALIDATION
# =============================================================================

print("="*70)
print("PREPROCESSING COMPLETE - FINAL VALIDATION")
print("="*70)

validation_passed = True

# 1. Shape consistency
print(f"\n1. Shape Consistency:")
print(f"   X_train:  {X.shape}")
print(f"   y_train:  {y.shape}")
print(f"   X_test:   {X_test.shape}")
print(f"   test_ids: {len(test_ids)}")

assert X.shape[0] == y.shape[0], "X and y row mismatch!"
assert X.shape[1] == X_test.shape[1], "Train/test feature mismatch!"
assert X_test.shape[0] == len(test_ids), "Test/IDs mismatch!"
print(f"   ✓ All shapes consistent")

# 2. Missing values
print(f"\n2. Missing Values:")
assert X.isnull().sum().sum() == 0, "Missing values in X!"
assert X_test.isnull().sum().sum() == 0, "Missing values in X_test!"
assert y.isnull().sum() == 0, "Missing values in y!"
print(f"   ✓ No missing values")

# 3. Infinite values
print(f"\n3. Infinite Values:")
assert not np.isinf(X.select_dtypes(include=[np.number])).any().any(), "Inf in X!"
assert not np.isinf(X_test.select_dtypes(include=[np.number])).any().any(), "Inf in X_test!"
print(f"   ✓ No infinite values")

# 4. Column consistency
print(f"\n4. Column Consistency:")
assert list(X.columns) == list(X_test.columns), "Column order mismatch!"
print(f"   ✓ Train and test have identical columns")

# 5. Data types
print(f"\n5. Data Types:")
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
assert len(non_numeric) == 0, f"Non-numeric columns: {non_numeric}"
print(f"   ✓ All features are numeric")

# 6. Target distribution
print(f"\n6. Target Distribution:")
print(f"   Mortality rate: {y.mean():.3f} ({y.sum()}/{len(y)})")
print(f"   Class 0: {(y==0).sum()}, Class 1: {(y==1).sum()}")

print(f"\n" + "="*70)
print(f"✅ ALL VALIDATION CHECKS PASSED")
print(f"="*70)
print(f"\nData is ready for modeling!")
print(f"  Training samples: {X.shape[0]:,}")
print(f"  Test samples:     {X_test.shape[0]:,}")
print(f"  Features:         {X.shape[1]}")

PREPROCESSING COMPLETE - FINAL VALIDATION

1. Shape Consistency:
   X_train:  (20885, 90)
   y_train:  (20885,)
   X_test:   (5221, 90)
   test_ids: 5221
   ✓ All shapes consistent

2. Missing Values:
   ✓ No missing values

3. Infinite Values:
   ✓ No infinite values

4. Column Consistency:
   ✓ Train and test have identical columns

5. Data Types:
   ✓ All features are numeric

6. Target Distribution:
   Mortality rate: 0.112 (2345/20885)
   Class 0: 18540, Class 1: 2345

✅ ALL VALIDATION CHECKS PASSED

Data is ready for modeling!
  Training samples: 20,885
  Test samples:     5,221
  Features:         90


---

## Preprocessing Summary

| Step | Description | Features Added/Modified |
|------|-------------|------------------------|
| 4.1 | Hospital History | 3 features (n_previous_icu_stays, is_first_icu_visit, is_frequent_flyer) |
| 4.2 | ICD9 Diagnosis | 9 features (n_diagnoses, disease_category, 7 condition flags) |
| 4.3 | Leakage Removal | Removed DEATHTIME, DOD, LOS, etc. |
| 4.4 | Age Calculation | 1 feature (age from DOB) |
| 4.5 | Imputation | Median (numeric), Mode (categorical) |
| 4.6 | Encoding | Target encoding + One-hot encoding |
| 4.7 | Feature Engineering | ~20 features (shock indices, clinical indicators, severity score) |
| 4.8 | Scaling | StandardScaler on continuous features only |

**Final Dataset:** 95 features, 20,885 training samples, 5,221 test samples

---

<a id='5-model-selection'></a>
## 5. Model Selection & Baseline

### Strategy
1. **Baseline Model:** Logistic Regression with class weights to handle imbalance
2. **Advanced Model:** Gradient Boosting Classifier - chosen for its:
   - Strong performance on tabular medical data
   - Built-in regularization through tree constraints
   - Ability to capture non-linear relationships
   - Less prone to overfitting compared to deeper models

### Evaluation Setup
- **Cross-Validation:** 5-fold Stratified K-Fold (preserves class imbalance)
- **Metric:** ROC-AUC (appropriate for imbalanced classification)

In [18]:
# =============================================================================
# 5.1 BASELINE MODEL - LOGISTIC REGRESSION
# =============================================================================

print("="*70)
print("5.1 BASELINE MODEL - LOGISTIC REGRESSION")
print("="*70)

# Set up cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Baseline: Logistic Regression with class weights
baseline_model = LogisticRegression(
    class_weight='balanced',  # Handle class imbalance
    max_iter=1000,
    random_state=RANDOM_STATE
)

# Cross-validation
baseline_scores = cross_val_score(
    baseline_model, X, y, 
    cv=cv, 
    scoring='roc_auc',
    n_jobs=-1
)

print(f"\nLogistic Regression (Baseline):")
print(f"  CV Scores: {[f'{s:.4f}' for s in baseline_scores]}")
print(f"  Mean ROC-AUC: {baseline_scores.mean():.4f} (+/- {baseline_scores.std()*2:.4f})")

# Fit for later comparison
baseline_model.fit(X, y)
baseline_train_auc = roc_auc_score(y, baseline_model.predict_proba(X)[:, 1])
print(f"  Training ROC-AUC: {baseline_train_auc:.4f}")
print(f"  Overfit gap: {baseline_train_auc - baseline_scores.mean():.4f}")

5.1 BASELINE MODEL - LOGISTIC REGRESSION

Logistic Regression (Baseline):
  CV Scores: ['0.8637', '0.8769', '0.8705', '0.8945', '0.8610']
  Mean ROC-AUC: 0.8733 (+/- 0.0239)
  Training ROC-AUC: 0.8802
  Overfit gap: 0.0069


In [19]:
# =============================================================================
# 5.2 MODEL COMPARISON - GRADIENT BOOSTING DEFAULT
# =============================================================================

print("\n" + "="*70)
print("5.2 GRADIENT BOOSTING - DEFAULT PARAMETERS")
print("="*70)

# Default Gradient Boosting
gb_default = GradientBoostingClassifier(
    random_state=RANDOM_STATE,
    n_estimators=100
)

# Cross-validation
gb_default_scores = cross_val_score(
    gb_default, X, y,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"\nGradient Boosting (Default):")
print(f"  CV Scores: {[f'{s:.4f}' for s in gb_default_scores]}")
print(f"  Mean ROC-AUC: {gb_default_scores.mean():.4f} (+/- {gb_default_scores.std()*2:.4f})")

# Compare to baseline
improvement = gb_default_scores.mean() - baseline_scores.mean()
print(f"\nImprovement over baseline: {improvement:+.4f} ({improvement/baseline_scores.mean()*100:+.2f}%)")
print(f"\n✓ Gradient Boosting shows strong potential - proceeding with hyperparameter tuning")


5.2 GRADIENT BOOSTING - DEFAULT PARAMETERS

Gradient Boosting (Default):
  CV Scores: ['0.8886', '0.8926', '0.8964', '0.9125', '0.8841']
  Mean ROC-AUC: 0.8948 (+/- 0.0194)

Improvement over baseline: +0.0216 (+2.47%)

✓ Gradient Boosting shows strong potential - proceeding with hyperparameter tuning


<a id='6-hyperparameter-tuning'></a>
## 6. Hyperparameter Tuning

### Approach: Optuna Bayesian Optimization

**Why Optuna over GridSearchCV?**
- **Efficiency:** Uses Bayesian optimization (TPE sampler) to intelligently explore the parameter space
- **Large search space:** GridSearch becomes infeasible with 7 hyperparameters
- **Early stopping:** Prunes unpromising trials automatically
- **Better convergence:** Typically finds better solutions in fewer trials

### Hyperparameter Search Space

| Parameter | Range | Rationale |
|-----------|-------|----------|
| n_estimators | 100-500 | More trees = lower bias, but slower |
| learning_rate | 0.01-0.15 | Lower = better generalization |
| max_depth | 2-6 | Shallow trees prevent overfitting |
| min_samples_split | 10-100 | Higher = more regularization |
| min_samples_leaf | 5-50 | Higher = smoother predictions |
| subsample | 0.5-0.95 | Stochastic gradient boosting |
| max_features | sqrt, log2, 0.5-0.9 | Feature subsampling |

### Optimization Objective
Maximize: `CV_mean - 0.5 * CV_std`

This penalizes high variance across folds, encouraging models that generalize well.

In [20]:
# =============================================================================
# 6.1 OPTUNA SETUP
# =============================================================================

# Install optuna if needed
try:
    import optuna
    print("✓ Optuna imported successfully")
except ImportError:
    print("Installing optuna...")
    !pip install optuna --quiet
    import optuna
    print("✓ Optuna installed and imported")

# Reduce Optuna verbosity
optuna.logging.set_verbosity(optuna.logging.WARNING)

✓ Optuna imported successfully


In [21]:
# =============================================================================
# 6.2 DEFINE OPTUNA OBJECTIVE FUNCTION
# =============================================================================

def objective(trial):
    """
    Optuna objective function for Gradient Boosting hyperparameter optimization.
    
    Objective: Maximize CV score while penalizing high variance
    This encourages models that generalize well to unseen data.
    """
    
    # Define hyperparameter search space
    params = {
        # Number of boosting stages (trees)
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        
        # Learning rate - lower values require more trees but generalize better
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        
        # Maximum depth of trees - shallow trees prevent overfitting
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        
        # Minimum samples required to split - regularization parameter
        'min_samples_split': trial.suggest_int('min_samples_split', 10, 100),
        
        # Minimum samples in leaf nodes - smoothing parameter
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 50),
        
        # Subsample ratio for stochastic gradient boosting
        'subsample': trial.suggest_float('subsample', 0.5, 0.95),
        
        # Number of features to consider for best split
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7, 0.9]),
        
        # Fixed parameters
        'random_state': RANDOM_STATE,
        'verbose': 0
    }
    
    # Create model
    model = GradientBoostingClassifier(**params)
    
    # 5-fold stratified cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    # Objective: Mean CV score penalized by variance
    # This encourages models with consistent performance across folds
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    score = cv_mean - 0.5 * cv_std
    
    return score

print("✓ Objective function defined")

✓ Objective function defined


In [23]:
# =============================================================================
# 6.3 RUN OPTUNA OPTIMIZATION
# =============================================================================

print("="*70)
print("OPTUNA HYPERPARAMETER OPTIMIZATION")
print("="*70)

# Configuration
N_TRIALS = 15  # Number of optimization trials

print(f"\nConfiguration:")
print(f"  Trials: {N_TRIALS}")
print(f"  CV folds: 5")
print(f"  Sampler: TPE (Tree-structured Parzen Estimator)")
print(f"  Objective: Maximize (CV_mean - 0.5 * CV_std)")
print(f"\n  Note: This may take 1-3 hours depending on hardware.")
print(f"        For faster results, reduce N_TRIALS to 30-50.")

# Progress callback
def progress_callback(study, trial):
    """Print progress every 10 trials."""
    if (trial.number + 1) % 10 == 0 or trial.number == 0:
        print(f"  Trial {trial.number + 1:3d}/{N_TRIALS}: "
              f"Score={trial.value:.4f} | Best={study.best_value:.4f}")

# Create Optuna study
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

print(f"\nStarting optimization...")
print("-"*70)

# Run optimization
study.optimize(
    objective,
    n_trials=N_TRIALS,
    callbacks=[progress_callback],
    show_progress_bar=False,
    n_jobs=1  # GB doesn't parallelize well within trials
)

print("-"*70)
print(f"\n✓ Optimization complete!")

OPTUNA HYPERPARAMETER OPTIMIZATION

Configuration:
  Trials: 15
  CV folds: 5
  Sampler: TPE (Tree-structured Parzen Estimator)
  Objective: Maximize (CV_mean - 0.5 * CV_std)

  Note: This may take 1-3 hours depending on hardware.
        For faster results, reduce N_TRIALS to 30-50.

Starting optimization...
----------------------------------------------------------------------
  Trial   1/15: Score=0.8936 | Best=0.8936


[W 2025-12-10 01:11:11,092] Trial 3 failed with parameters: {'n_estimators': 480, 'learning_rate': 0.13666943328202277, 'max_depth': 6, 'min_samples_split': 37, 'min_samples_leaf': 9, 'subsample': 0.8079048619304705, 'max_features': 0.9} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Code\BSEcode\Computational Machine Learning\CML_final_project_2025\Classification _HEF\.venv\Lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\corne\AppData\Local\Temp\ipykernel_29444\3709781036.py", line 46, in objective
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
  File "c:\Code\BSEcode\Computational Machine Learning\CML_final_project_2025\Classification _HEF\.venv\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "c:\Code\BSEcode\Computational Machine Learning\CML_final_project_20

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# 6.4 OPTUNA RESULTS ANALYSIS
# =============================================================================

print("="*70)
print("OPTIMIZATION RESULTS")
print("="*70)

print(f"\nBest Trial: #{study.best_trial.number}")
print(f"Best Score (CV - 0.5*std): {study.best_value:.4f}")

print(f"\nBest Hyperparameters:")
best_params = study.best_params
for param, value in best_params.items():
    if isinstance(value, float):
        print(f"  {param:<20} {value:.4f}")
    else:
        print(f"  {param:<20} {value}")

# Top 5 trials
print(f"\nTop 5 Trials:")
print(f"  {'Trial':<8} {'Score':<10} {'n_est':<8} {'lr':<8} {'depth':<6}")
print(f"  {'-'*44}")

top_trials = sorted(study.trials, key=lambda t: t.value if t.value else 0, reverse=True)[:5]
for trial in top_trials:
    if trial.value:
        print(f"  {trial.number:<8} {trial.value:<10.4f} "
              f"{trial.params['n_estimators']:<8} "
              f"{trial.params['learning_rate']:<8.4f} "
              f"{trial.params['max_depth']:<6}")

# Parameter importance
if len(study.trials) >= 20:
    print(f"\nParameter Importance (from Optuna):")
    try:
        importance = optuna.importance.get_param_importances(study)
        for param, imp in sorted(importance.items(), key=lambda x: x[1], reverse=True):
            print(f"  {param:<20} {imp:.3f}")
    except:
        print("  (Could not compute importance)")

<a id='7-final-model'></a>
## 7. Final Model & Evaluation

Using the best hyperparameters from Optuna optimization, we train the final model on the full training dataset.

In [ ]:
# =============================================================================
# 7.1 TRAIN FINAL MODEL
# =============================================================================

print("="*70)
print("TRAINING FINAL MODEL")
print("="*70)

# Prepare final parameters
final_params = best_params.copy()
final_params.update({
    'random_state': RANDOM_STATE,
    'verbose': 0
})

print(f"\nFinal Model Parameters:")
for param, value in final_params.items():
    if isinstance(value, float):
        print(f"  {param}: {value:.4f}")
    else:
        print(f"  {param}: {value}")

# Create final model
final_model = GradientBoostingClassifier(**final_params)

# Final cross-validation with best parameters
print(f"\nFinal Cross-Validation:")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
final_cv_scores = cross_val_score(final_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"  Fold scores: {[f'{s:.4f}' for s in final_cv_scores]}")
print(f"  Mean ROC-AUC: {final_cv_scores.mean():.4f}")
print(f"  Std: {final_cv_scores.std():.4f}")

# Train on full dataset
print(f"\nTraining on full dataset ({len(X):,} samples)...")
final_model.fit(X, y)
print("✓ Training complete")

In [ ]:
# =============================================================================
# 7.2 OVERFITTING ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("OVERFITTING ANALYSIS")
print("="*70)

# Training performance
train_proba = final_model.predict_proba(X)[:, 1]
train_auc = roc_auc_score(y, train_proba)

print(f"\nPerformance Comparison:")
print(f"  Training ROC-AUC:     {train_auc:.4f}")
print(f"  CV ROC-AUC (Mean):    {final_cv_scores.mean():.4f}")
print(f"  CV ROC-AUC (Std):     {final_cv_scores.std():.4f}")

gap = train_auc - final_cv_scores.mean()
print(f"\n  Train-CV Gap: {gap:.4f}")

if gap < 0.03:
    print(f"  Assessment: ✓ Excellent - Minimal overfitting")
elif gap < 0.05:
    print(f"  Assessment: ✓ Good - Low overfitting")
elif gap < 0.10:
    print(f"  Assessment: ⚠ Moderate overfitting")
else:
    print(f"  Assessment: ❌ High overfitting - consider more regularization")

# Compare to baseline
print(f"\nImprovement over Baseline:")
print(f"  Baseline (Logistic Regression): {baseline_scores.mean():.4f}")
print(f"  Final Model (Gradient Boosting): {final_cv_scores.mean():.4f}")
print(f"  Improvement: {final_cv_scores.mean() - baseline_scores.mean():+.4f} "
      f"({(final_cv_scores.mean() - baseline_scores.mean())/baseline_scores.mean()*100:+.2f}%)")

In [ ]:
# =============================================================================
# 7.3 FEATURE IMPORTANCE ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("FEATURE IMPORTANCE (TOP 20)")
print("="*70)

# Get feature importances
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

# Display top 20
print(f"\n{'Rank':<6} {'Feature':<30} {'Importance':<12}")
print("-"*50)
for i, (_, row) in enumerate(feature_importance.head(20).iterrows(), 1):
    print(f"{i:<6} {row['feature']:<30} {row['importance']:.4f}")

# Visualize top 15
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'], color='steelblue')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance (Gini)')
plt.title('Top 15 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 7.4 ROC CURVE VISUALIZATION
# =============================================================================

print("\n" + "="*70)
print("ROC CURVE ANALYSIS")
print("="*70)

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y, train_proba)

# Find optimal threshold (Youden's J statistic)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"\nOptimal Threshold Analysis:")
print(f"  Optimal threshold (Youden's J): {optimal_threshold:.4f}")
print(f"  True Positive Rate at optimal: {tpr[optimal_idx]:.4f}")
print(f"  False Positive Rate at optimal: {fpr[optimal_idx]:.4f}")

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC curve (AUC = {train_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
plt.scatter([fpr[optimal_idx]], [tpr[optimal_idx]], color='red', s=100, 
            zorder=5, label=f'Optimal threshold ({optimal_threshold:.3f})')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Gradient Boosting Classifier')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id='8-predictions'></a>
## 8. Predictions Generation

Generate probability predictions for the test set and save in the required format.

In [ ]:
# =============================================================================
# 8.1 GENERATE TEST PREDICTIONS
# =============================================================================

print("="*70)
print("GENERATING TEST PREDICTIONS")
print("="*70)

# Generate probability predictions
test_proba = final_model.predict_proba(X_test)[:, 1]

print(f"\nPrediction Statistics:")
print(f"  Samples: {len(test_proba):,}")
print(f"  Min probability: {test_proba.min():.4f}")
print(f"  Max probability: {test_proba.max():.4f}")
print(f"  Mean probability: {test_proba.mean():.4f}")
print(f"  Median probability: {np.median(test_proba):.4f}")

# Distribution analysis
print(f"\nPrediction Distribution:")
bins = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.0]
for i in range(len(bins)-1):
    count = ((test_proba >= bins[i]) & (test_proba < bins[i+1])).sum()
    pct = count / len(test_proba) * 100
    bar = '█' * int(pct / 2)
    print(f"  {bins[i]:.2f}-{bins[i+1]:.2f}: {count:5d} ({pct:5.1f}%) {bar}")

In [ ]:
# =============================================================================
# 8.2 VALIDATE AND SAVE PREDICTIONS
# =============================================================================

print("\n" + "="*70)
print("SAVING PREDICTIONS")
print("="*70)

# Create submission dataframe
submission = pd.DataFrame({
    'icustay_id': test_ids,
    'prediction': test_proba
})

# Validation checks
print(f"\nValidation:")
assert len(submission) == len(X_test), "Length mismatch!"
print(f"  ✓ Correct number of predictions: {len(submission):,}")

assert submission['prediction'].notna().all(), "NaN predictions found!"
print(f"  ✓ No missing predictions")

assert (submission['prediction'] >= 0).all() and (submission['prediction'] <= 1).all(), "Invalid probabilities!"
print(f"  ✓ All predictions in valid range [0, 1]")

assert submission['icustay_id'].nunique() == len(submission), "Duplicate IDs!"
print(f"  ✓ All ICU stay IDs are unique")

# Save to CSV (assignment format)
output_filename = 'vandenbosch_corneel_CML_2025.csv'
submission.to_csv(output_filename, index=False)

print(f"\n✓ Predictions saved to: {output_filename}")
print(f"\nSubmission preview:")
print(submission.head(10).to_string(index=False))

In [ ]:
# =============================================================================
# 8.3 PREDICTION VISUALIZATION
# =============================================================================

# Visualize prediction distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(test_proba, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(test_proba.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {test_proba.mean():.3f}')
axes[0].axvline(np.median(test_proba), color='orange', linestyle='--', linewidth=2,
                label=f'Median: {np.median(test_proba):.3f}')
axes[0].set_xlabel('Predicted Probability of Mortality')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Test Predictions')
axes[0].legend()

# Compare with training predictions
axes[1].hist(train_proba, bins=50, alpha=0.5, label='Training', color='blue')
axes[1].hist(test_proba, bins=50, alpha=0.5, label='Test', color='orange')
axes[1].set_xlabel('Predicted Probability of Mortality')
axes[1].set_ylabel('Count')
axes[1].set_title('Training vs Test Prediction Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nNote: Similar distributions suggest the model generalizes well.")

<a id='9-conclusion'></a>
## 9. Conclusion

### Summary

This project developed a binary classification model to predict in-hospital mortality for ICU patients using the MIMIC-III dataset.

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("="*70)
print("PROJECT SUMMARY")
print("="*70)

print(f"\n📊 DATASET:")
print(f"   Training samples: {len(X):,}")
print(f"   Test samples: {len(X_test):,}")
print(f"   Features: {X.shape[1]}")
print(f"   Mortality rate: {y.mean()*100:.1f}%")

print(f"\n🔧 PREPROCESSING HIGHLIGHTS:")
print(f"   • Hospital history features (repeat visits, frequent flyers)")
print(f"   • ICD9 diagnosis encoding (target encoding + disease categories)")
print(f"   • 7 high-risk condition flags (sepsis, heart failure, etc.)")
print(f"   • Medical severity indicators (shock index, vital sign ranges)")
print(f"   • Proper handling of MIMIC-III age shifts")

print(f"\n🤖 MODEL:")
print(f"   Algorithm: Gradient Boosting Classifier (sklearn)")
print(f"   Optimization: Optuna Bayesian optimization ({N_TRIALS} trials)")

print(f"\n📈 PERFORMANCE:")
print(f"   Baseline (Logistic Regression): {baseline_scores.mean():.4f} ROC-AUC")
print(f"   Final Model CV Score: {final_cv_scores.mean():.4f} ROC-AUC")
print(f"   Improvement: {(final_cv_scores.mean() - baseline_scores.mean())*100:.2f}%")

print(f"\n🎯 KEY HYPERPARAMETERS:")
print(f"   n_estimators: {best_params['n_estimators']}")
print(f"   learning_rate: {best_params['learning_rate']:.4f}")
print(f"   max_depth: {best_params['max_depth']}")
print(f"   subsample: {best_params['subsample']:.4f}")

print(f"\n📁 OUTPUT:")
print(f"   Predictions file: {output_filename}")
print(f"   Format: icustay_id, prediction (probabilities)")

print(f"\n" + "="*70)
print("END OF NOTEBOOK")
print("="*70)

---

### Key Design Decisions

| Decision | Rationale |
|----------|----------|
| **Gradient Boosting over Random Forest** | Better performance on this medical dataset; more control over regularization |
| **Optuna over GridSearchCV** | Efficient Bayesian search over large parameter space |
| **Target encoding for ICD9 codes** | Handles 1,800+ unique values without explosion of features |
| **Shock indices as features** | Clinically validated predictors of ICU mortality |
| **CV-variance penalization** | Encourages models that generalize to unseen data |
| **Stratified K-Fold** | Preserves class imbalance in each fold |
| **Predict probabilities** | Required by assignment; more informative than binary predictions |

### Most Important Features

1. **primary_diag_encoded**: Primary diagnosis (target encoded)
2. **ICD9_encoded**: ICD9 diagnosis code (target encoded)
3. **Temperature features**: TempC_Mean, TempC_Min, TempC_Max
4. **Blood pressure**: SysBP_Min, SysBP_Mean
5. **Oxygen saturation**: SpO2_Min, SpO2_Mean
6. **High-risk conditions**: has_respiratory_failure, has_sepsis, has_aki

### Potential Improvements

1. **Ensemble of models**: Combine GB with XGBoost, LightGBM
2. **More clinical features**: Lab values, medications if available
3. **Time-series features**: Trends in vital signs over time
4. **External validation**: Test on different hospital datasets

---

*Notebook created for CML Final Project 2025*